# Exploring the capstone calls

A quick look at the Mutect2 results for chr22:34127223-34838003:
what was found and missed, which kinds of mutations, and why Mutect2 rejected its other candidates.


In [ ]:
import gzip
from pathlib import Path
import pandas as pd

# The project folder: works whether Jupyter runs from the project or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "pixi.toml").exists() else Path.cwd().parent

summary = pd.read_csv(ROOT / "results/capstone/comparison/summary.tsv", sep="\t")
table = pd.read_csv(ROOT / "results/capstone/comparison/comparison.tsv", sep="\t")
summary

In [ ]:
table["type"] = ["SNV" if len(r) == 1 and len(a) == 1 else "indel"
                 for r, a in zip(table["ref"], table["alt"])]
pd.crosstab(table["type"], table["status"])

In [ ]:
rows = []
with gzip.open(ROOT / "results/capstone/calls/filtered.vcf.gz", "rt") as handle:
    for line in handle:
        if line.startswith("#"):
            continue
        f = line.split("\t")
        rows.append({"pos": int(f[1]) + 34127222, "ref": f[3], "alt": f[4], "filter": f[6]})
calls = pd.DataFrame(rows)

print(len(calls), "candidates,", (calls["filter"] == "PASS").sum(), "PASS")
calls["filter"].str.split(";").explode().value_counts()

In [ ]:
with gzip.open(ROOT / "results/capstone/calls/filtered.vcf.gz", "rt") as handle:
    for line in handle:
        if line.startswith("##FILTER"):
            print(line.strip())

## Notes
- Mutect2 found all 5 known single-letter changes (SNVs) in this stretch, but only 2 of the 8 known insertions/deletions.
- None of the 6 missed mutations was filtered out: Mutect2 never proposed them as candidates.
- Mutect2 proposed 193 candidates and its filters kept 8. Most rejected candidates had several problems at once, most often strand bias, weak evidence and an artifact also seen in the normal.